# LSTM — hyperparameter search (Weights & Biases)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rokhaya8/fall-detection-lstm/blob/main/notebooks/03_lstm_hyperparameter_search.ipynb)

Part of [fall-detection-lstm](https://github.com/Rokhaya8/fall-detection-lstm). Paths are set by `BASE_DIR` in the first code cell — adapt it to your Google Drive.

# Recherche des Hyperparamètres - Modèle LSTM

**Méthode** : Random Search via Weights & Biases (W&B)

#Installation de Weights & Biases

Installation de la bibliothèque W&B nécessaire pour le sweep d'hyperparamètres.
À exécuter une seule fois par session Colab.

In [ ]:
!pip install wandb -Uq

#Montage du Drive
Monte Google Drive pour accéder aux données préprocessées
(X_train, X_val, y_train, y_val) nécessaires à l'entraînement.

In [ ]:
# 1. Montage du drive
from google.colab import drive
drive.mount('/content/drive')

# ── Dossier du projet dans Google Drive (à adapter si besoin) ──
import os
BASE_DIR = '/content/drive/MyDrive/Projet_Parkinson'
os.makedirs(f'{BASE_DIR}/Figures', exist_ok=True)


Mounted at /content/drive


## Cellule 3 — Chargement des données préprocessées

Chargement des tenseurs SisFall préprocessés depuis Google Drive.
Vérification des shapes et du ratio chutes/activités sur le train set.

In [ ]:
import numpy as np
# Charger les données
X_train = np.load(f'{BASE_DIR}/Data_Preprocessed/SisFall/X_train.npy')
X_val   = np.load(f'{BASE_DIR}/Data_Preprocessed/SisFall/X_val.npy')
X_test  = np.load(f'{BASE_DIR}/Data_Preprocessed/SisFall/X_test.npy')
y_train = np.load(f'{BASE_DIR}/Data_Preprocessed/SisFall/y_train.npy')
y_val   = np.load(f'{BASE_DIR}/Data_Preprocessed/SisFall/y_val.npy')
y_test  = np.load(f'{BASE_DIR}/Data_Preprocessed/SisFall/y_test.npy')

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"Ratio chutes train : {y_train.mean()*100:.1f}%")

X_train : (123031, 200, 6)
X_val   : (15190, 200, 6)
X_test  : (20367, 200, 6)
y_train : (123031,)
Ratio chutes train : 34.7%


#Authentification W&B

Connexion au compte Weights & Biases.

Une clé API est requise pour envoyer les résultats des runs
sur le tableau de bord W&B en ligne.

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rokhayagueye8 (rokhayagueye8-national-school-of-applied-sciences-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

#Initialisation de la configuration du sweep

Création du dictionnaire de configuration du sweep avec la méthode
**random search** (recherche aléatoire). Cette méthode tire aléatoirement
des combinaisons d'hyperparamètres parmi les valeurs candidates, ce qui
est plus efficace qu'une recherche exhaustive (grid search) pour un espace
de 108 combinaisons possibles.

In [ ]:
# Étape 1 : Initialisation de la configuration avec la méthode choisie
sweep_config = {
    'method': 'random'  # Le robot testera les combinaisons de manière aléatoire
}

#Définition de la métrique et des hyperparamètres à explorer

**Métrique optimisée :** `val_loss` (minimisation) - garantit que le modèle
retenu généralise bien sur des données non vues pendant l'entraînement.

**Arrêt anticipé Hyperband** (`min_iter=10, eta=2`) - élimine les runs peu
prometteurs après 10 epochs minimum, réduisant le temps total du sweep.

**Hyperparamètres explorés :**

| Hyperparamètre | Valeurs testées | Justification |
|---|---|---|
| lstm_layers | 1, 2 | Tester si une 2ème couche améliore les performances |
| lstm_units | 32, 64, 128 | Tester différentes capacités de mémorisation |
| dropout | 0.2, 0.3, 0.4 | Tester différents niveaux de régularisation |
| learning_rate | 0.001, 0.0001 | Tester des vitesses d'apprentissage différentes |
| batch_size | 64, 128, 256 | Tester l'impact de la taille des mini-lots |

In [ ]:
# 1. Définition de la métrique (comme dans le tuto)
metric = {
    'name': 'val_loss', # On utilise 'val_loss' (perte de validation) pour éviter le surapprentissage
    'goal': 'minimize'
}
sweep_config['metric'] = metric

# 2. Adaptation de vos hyperparamètres spécifiques à votre LSTM Keras
parameters_dict = {
    # Nous voulons tester si 1 ou 2 couches LSTM c'est mieux
    'lstm_layers': {
        'values':  [1, 2]
    },
    # Nous voulons tester le nombre d'unités (la mémoire du LSTM)
    'lstm_units': {
        'values': [32, 64, 128]
    },
    # Votre dropout actuel est à 0.3, on va tester autour de cette valeur
    'dropout': {
        'values': [0.2, 0.3, 0.4]
    },
    # On ajoute le taux d'apprentissage de l'optimiseur Adam
    'learning_rate': {
        'values': [0.001, 0.0001]
    },
    # Votre batch_size actuel est de 32, on teste d'autres valeurs standards
    'batch_size': {
        'values': [64, 128, 256]
    }
}

# (optionnel mais recommandé pour gagner du temps)
sweep_config['early_terminate'] = {'type': 'hyperband', 'min_iter': 10, 'eta': 2}

sweep_config['parameters'] = parameters_dict


#Ajout du nombre d'epochs maximum

Fixe le nombre maximum d'epochs à **100** pour chaque run.
C'est un plafond - l'arrêt réel de chaque run est géré par
l'`EarlyStopping(patience=10)` dans la fonction d'entraînement.

In [ ]:
# On valide la configuration avec un maximum de 100 époques pour aller plus vite
parameters_dict.update({ #update c'es pour ajouter à un sweep que l'on a déjà initialiser
    'epochs': {
        'value': 100
    }
})


#Vérification de la configuration du sweep

Affiche la configuration complète du sweep de façon lisible
pour vérifier que tous les paramètres sont correctement définis
avant d'envoyer la configuration aux serveurs W&B.

In [ ]:
import pprint
pprint.pprint(sweep_config)

{'early_terminate': {'eta': 2, 'min_iter': 10, 'type': 'hyperband'},
 'method': 'random',
 'metric': {'goal': 'minimize', 'name': 'val_loss'},
 'parameters': {'batch_size': {'values': [64, 128, 256]},
                'dropout': {'values': [0.2, 0.3, 0.4]},
                'epochs': {'value': 100},
                'learning_rate': {'values': [0.001, 0.0001]},
                'lstm_layers': {'values': [1, 2]},
                'lstm_units': {'values': [32, 64, 128]}}}


#Création du sweep sur W&B

Envoie la configuration du sweep aux serveurs W&B et crée
un sweep identifié par un ID unique (`sweep_id`).
Le projet **"parkinson-lstm-sweep"** regroupe tous les runs
sur le tableau de bord W&B en ligne pour visualisation et comparaison.

Le `sweep_id` est nécessaire pour lancer l'agent dans la cellule suivante il doit rester en mémoire pour toute la session.

In [ ]:
# ÉTAPE 2 : On envoie votre configuration finale aux serveurs de W&B
sweep_id = wandb.sweep(sweep_config, project="parkinson-lstm-sweep")

Create sweep with ID: mniuyfmn
Sweep URL: https://wandb.ai/rokhayagueye8-national-school-of-applied-sciences-/parkinson-lstm-sweep/sweeps/mniuyfmn


#Cellule 9 — Définition des fonctions d'entraînement

Cette cellule définit les trois fonctions appelées par l'agent W&B
à chaque run du sweep :

**`build_network(lstm_layers, lstm_units, dropout)`**
Construit l'architecture du modèle LSTM selon les hyperparamètres
tirés par le sweep. Si `lstm_layers=2`, une deuxième couche LSTM
est ajoutée avec `return_sequences=True` sur la première couche.

**`compile_model(model, learning_rate)`**
Compile le modèle avec l'optimiseur Adam et la fonction de perte
Binary Crossentropy. Les métriques suivies sont l'accuracy,
le recall et la precision.

**`train(config=None)`**
Fonction principale appelée par l'agent à chaque run. Elle :
- Récupère les hyperparamètres choisis par W&B (`wandb.config`)
- Construit et compile le modèle
- Configure l'`EarlyStopping` (patience=10) pour éviter le surapprentissage
- Lance l'entraînement avec `class_weight={0:1.0, 1:3.0}` pour
  compenser le déséquilibre des classes
- Envoie les métriques à W&B via `WandbMetricsLogger` à chaque epoch

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Nouvelle méthode d'importation officielle et sécurisée de W&B pour Keras
from wandb.integration.keras import WandbMetricsLogger


# --- ÉQUIVALENT DE BUILD_NETWORK ---
# Cette fonction crée l'architecture de votre LSTM en fonction des choix du Sweep
def build_network(lstm_layers, lstm_units, dropout):
    model = Sequential()

    # Première couche LSTM
    return_seq = (lstm_layers > 1) # True s'il y a une deuxième couche après
    model.add(LSTM(
        units=lstm_units,
        input_shape=(200, 6),
        return_sequences=return_seq
    ))
    model.add(Dropout(dropout))

    # Deuxième couche LSTM (optionnelle)
    if lstm_layers > 1:
        model.add(LSTM(units=lstm_units, return_sequences=False))
        model.add(Dropout(dropout))

    model.add(Dense(1, activation='sigmoid'))
    return model

# --- ÉQUIVALENT DE BUILD_OPTIMIZER & COMPILATION ---
# Cette fonction configure l'optimiseur Adam avec le bon taux d'apprentissage
def compile_model(model, learning_rate):
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Recall(), tf.keras.metrics.Precision()]
    )
    return model

# --- LA FONCTION PRINCIPALE : TRAIN ---
# C'est la fonction maîtresse que le Sweep appellera en boucle
def train(config=None):
    # Initialize a new wandb run
    with wandb.init(config=config) as run:
        # Récupération des hyperparamètres choisis pour cette session de test
        config = wandb.config

        # 1. Construction du réseau avec les paramètres du Sweep
        network = build_network(config.lstm_layers, config.lstm_units, config.dropout)

        # 2. Compilation du réseau avec le learning rate du Sweep
        network = compile_model(network, config.learning_rate)

        # 3. Configuration de vos garde-fous (Callbacks)
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        )

        # 4. Entraînement complet (Équivalent de la boucle d'époque PyTorch)
        # Keras gère la boucle d'époque automatiquement avec model.fit()
        network.fit(
            X_train, y_train,
            epochs=config.epochs,         # Reçoit vos 100 époques max
            batch_size=config.batch_size, # Reçoit 16, 32, ou 64
            validation_data=(X_val, y_val),
            # Le WandbMetricsLogger remplace le "wandb.log" manuel du tutoriel
            callbacks=[early_stopping, WandbMetricsLogger()],
            class_weight={0: 1.0, 1: 3.0}
        )


#Cellule 10 — Lancement de l'agent W&B

Lance l'agent W&B qui orchestre le sweep en exécutant **15 runs**
automatiquement (`count=15`). À chaque run, l'agent tire aléatoirement
une combinaison d'hyperparamètres, appelle la fonction `train()`,
récupère la `val_loss` finale et décide du run suivant.

L'agent s'arrête automatiquement après 15 runs ou si
l'`early_terminate` Hyperband juge que suffisamment de combinaisons
ont été explorées. Les résultats de chaque run sont visibles
en temps réel sur le tableau de bord W&B.

In [ ]:
# ÉTAPE 4 : Activation de l'agent pour lancer 15 entraînements automatiques
wandb.agent(sweep_id, function=train, count=15)


wandb: Agent Starting Run: akd8nt6i with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.3
wandb: 	epochs: 100
wandb: 	learning_rate: 0.001
wandb: 	lstm_layers: 1
wandb: 	lstm_units: 32
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 30s 13ms/step - accuracy: 0.7290 - loss: 0.8986 - precision: 0.5996 - recall: 0.6628 - val_accuracy: 0.8264 - val_loss: 0.5783 - val_precision: 0.7210 - val_recall: 0.6753
Epoch 2/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 26s 13ms/step - accuracy: 0.7796 - loss: 0.8515 - precision: 0.6892 - recall: 0.6662 - val_accuracy: 0.8019 - val_loss: 0.5603 - val_precision: 0.6495 - val_recall: 0.7196
Epoch 3/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 26s 13ms/step - accuracy: 0.7757 - loss: 0.8327 - precision: 0.6736 - recall: 0.6876 - val_accuracy: 0.7924 - val_loss: 0.5187 - val_precision: 0.6259 - val_recall: 0.7440
Epoch 4/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - accuracy: 0.7454 - loss: 0.8658 - precision: 0.6221 - recall: 0.6806 - val_accuracy: 0.8386 - val_loss: 0.5433 - val_precision: 0.7452 - val_recall: 0.6920
Epoch 5/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 26s 13ms/step - accuracy: 0.7847 - loss: 0.8504 - precision: 0.7010 - recall: 0.6632 - val_accura

epoch/accuracy,▂▇▇▃█▆▅▃▃▂▃▂▃▅▅▃▄▆▆▃▃▁▄▅▄▅▆▆▆▇▇▇█
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▆▆▅▅▅▅▆▅▅▄▅▄▅▅▅▄▆▅▅▃▄▃▃▂▃▂▂▂▁▁
epoch/precision,▂▇▆▃█▆▄▃▂▂▃▂▃▄▄▃▃▆▅▃▃▁▃▄▃▄▄▄▄▅▅▅▆
epoch/recall,▁▁▂▂▁▃▄▅▄▄▄▅▅▄▆▄▄▃▄▄▅▆▆▄▇▇▇▇▇▇███
epoch/val_accuracy,▇▆▅█▇▆▅▆▄▆▁▄▇▅▆▇▄▇▆▇▃▆▆▆▅▆▄▄▄▄▅▅▅
epoch/val_loss,▇▆▄▅▅▅▅▃▆▆▇▄▂▆█▃▆▄▃▅▅▄▁▄▃▂▃▃▃▃▂▃▃
epoch/val_precision,▇▅▄█▆▅▄▅▃▅▁▃▇▄▅▇▃▆▆▆▂▆▅▅▄▅▃▃▃▃▄▄▄
epoch/val_recall,▁▃▄▂▃▃▆▂▃▁▆▄▃▆▂▃▄▃▃▃▅▃▂▄▇▅▆▆▆█▇▇█
epoch/accuracy,0.78912


wandb: Agent Starting Run: pp1edr8m with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.2
wandb: 	epochs: 100
wandb: 	learning_rate: 0.0001
wandb: 	lstm_layers: 2
wandb: 	lstm_units: 64
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

Epoch 1/100
481/481 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.7019 - loss: 0.9420 - precision: 0.5608 - recall: 0.6548 - val_accuracy: 0.7935 - val_loss: 0.5984 - val_precision: 0.6497 - val_recall: 0.6571
Epoch 2/100
481/481 ━━━━━━━━━━━━━━━━━━━━ 13s 28ms/step - accuracy: 0.7806 - loss: 0.8553 - precision: 0.6967 - recall: 0.6526 - val_accuracy: 0.8377 - val_loss: 0.5295 - val_precision: 0.7430 - val_recall: 0.6913
Epoch 3/100
481/481 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.7882 - loss: 0.8280 - precision: 0.7057 - recall: 0.6700 - val_accuracy: 0.8207 - val_loss: 0.5455 - val_precision: 0.6881 - val_recall: 0.7222
Epoch 4/100
481/481 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.7852 - loss: 0.8170 - precision: 0.6954 - recall: 0.6795 - val_accuracy: 0.8144 - val_loss: 0.5548 - val_precision: 0.6780 - val_recall: 0.7116
Epoch 5/100
481/481 ━━━━━━━━━━━━━━━━━━━━ 13s 28ms/step - accuracy: 0.7701 - loss: 0.8005 - precision: 0.6537 - recall: 0.7195 - val_accuracy: 0.8207

epoch/accuracy,▁▇██▇▆▅▆▆▆▇▇▇▇▇▇█▇▇
epoch/epoch,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
epoch/precision,▁███▅▄▄▄▄▄▅▅▄▅▅▅▅▅▅
epoch/recall,▁▁▂▂▃▅▆▇▇▇▇████████
epoch/val_accuracy,▆█▇▇▇▃▃▅▇▄▄▅▄▁▄▆▄▄▃
epoch/val_loss,▇▄▅▅▁▅▅▂▁▅▄▂▄█▃▁▃▃▅
epoch/val_precision,▅█▇▆▆▂▂▄▅▃▃▄▃▁▃▅▃▃▂
epoch/val_recall,▁▂▃▃▄█▇▆▅▇▇▅▇█▇▆▆▇▆
epoch/accuracy,0.78035


wandb: Agent Starting Run: 1qlq0gem with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.2
wandb: 	epochs: 100
wandb: 	learning_rate: 0.001
wandb: 	lstm_layers: 2
wandb: 	lstm_units: 32
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 50s 25ms/step - accuracy: 0.7560 - loss: 0.8794 - precision: 0.6491 - recall: 0.6483 - val_accuracy: 0.7709 - val_loss: 0.6061 - val_precision: 0.6001 - val_recall: 0.6796
Epoch 2/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 45s 23ms/step - accuracy: 0.7553 - loss: 0.8679 - precision: 0.6409 - recall: 0.6728 - val_accuracy: 0.7769 - val_loss: 0.6104 - val_precision: 0.6185 - val_recall: 0.6444
Epoch 3/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 46s 24ms/step - accuracy: 0.7047 - loss: 0.8921 - precision: 0.5598 - recall: 0.7024 - val_accuracy: 0.8239 - val_loss: 0.5339 - val_precision: 0.7103 - val_recall: 0.6849
Epoch 4/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 46s 24ms/step - accuracy: 0.7247 - loss: 0.8706 - precision: 0.5878 - recall: 0.6956 - val_accuracy: 0.8234 - val_loss: 0.5607 - val_precision: 0.7146 - val_recall: 0.6727
Epoch 5/100
1923/1923 ━━━━━━━━━━━━━━━━━━━━ 45s 24ms/step - accuracy: 0.7150 - loss: 0.8597 - precision: 0.5707 - recall: 0.7252 - val_accura

wandb: Ctrl + C detected. Stopping sweep.
